# 📐 02 · Baseline — Regresión Logística

**Proyecto:** Detección de fraude transaccional en tiempo real
**Autoras:** Gabriela Cabrera · Jessica Rivera

Construimos un baseline simple usando **Regresión Logística con balanced class weight** como punto de comparación mínimo, conforme al requisito del Corte 3.

**Objetivo:** establecer una línea base honesta antes de pasar al modelo final XGBoost en `03_modelo_final.ipynb`.

In [ ]:
# Instalación de dependencias en Colab (~10 seg)
!pip install -q xgboost scikit-learn

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, roc_curve, precision_recall_curve,
    confusion_matrix
)
import warnings; warnings.filterwarnings("ignore")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.dpi"] = 110
print("✔ Imports listos.")

## 1. Construcción del dataset con 17 features tipo IEEE-CIS

In [ ]:
def construir_dataset(N=200_000, seed=RANDOM_SEED):
    """Dataset sintético con 17 features inspiradas en IEEE-CIS."""
    rng = np.random.default_rng(seed)
    TransactionDT = np.sort(rng.uniform(0, 6*30*86400, N))
    isFraud = rng.binomial(1, 0.035, N)

    # Monto: lognormal por clase
    TransactionAmt = np.where(isFraud == 1,
                               rng.lognormal(4.7, 1.2, N),
                               rng.lognormal(4.0, 0.9, N))

    df = pd.DataFrame({
        "TransactionDT": TransactionDT,
        "TransactionAmt": TransactionAmt,
        "log_amt": np.log1p(TransactionAmt),
        "amt_round": (TransactionAmt == np.round(TransactionAmt)).astype(int),
        "amt_decimals": (np.round(TransactionAmt * 100) % 100 == 0).astype(int),
        "small_amount": (TransactionAmt < 10).astype(int),
        "hour": ((TransactionDT / 3600) % 24).astype(int),
        "day_of_week": ((TransactionDT / 86400) % 7).astype(int),
        "card_tipo": rng.integers(0, 4, N),
        "product_cd": rng.integers(0, 5, N),
        "pais_riesgo": np.where(isFraud == 1,
                                  rng.choice([3,4,5,5], N),
                                  rng.choice([1,1,2,2,3], N)),
        "email_risk": np.where(isFraud == 1,
                                rng.choice([2,3,4,5,5], N),
                                rng.choice([1,1,1,2,3], N)),
        "device_type": rng.integers(0, 2, N),
        "device_os_risk": np.where(isFraud == 1,
                                     rng.choice([2,3,3,4], N),
                                     rng.choice([1,1,2,2], N)),
        "browser_risk": rng.integers(1, 5, N),
        "billing_ship_match": np.where(isFraud == 1,
                                         rng.choice([0,0,0,1], N),
                                         rng.choice([0,1,1,1], N)),
        "prev_attempts_24h": np.where(isFraud == 1,
                                        rng.poisson(3, N),
                                        rng.poisson(0.4, N)),
        "ip_billing_distance": np.where(isFraud == 1,
                                          rng.exponential(4000, N),
                                          rng.exponential(200, N)),
        "isFraud": isFraud,
    }).sort_values("TransactionDT").reset_index(drop=True)

    return df

df = construir_dataset()
print(f"Dataset: {df.shape}")
print(f"Tasa de fraude: {df['isFraud'].mean()*100:.3f}%")
df.head()

## 2. Partición temporal estricta (70/15/15 sin shuffle)

In [ ]:
FEATURES = [
    "TransactionAmt", "log_amt", "amt_round", "amt_decimals", "small_amount",
    "hour", "day_of_week", "card_tipo", "product_cd", "pais_riesgo",
    "email_risk", "device_type", "device_os_risk", "browser_risk",
    "billing_ship_match", "prev_attempts_24h", "ip_billing_distance",
]

N = len(df)
n_train = int(N * 0.70)
n_val = int(N * 0.15)

X_train = df[FEATURES].iloc[:n_train]
y_train = df["isFraud"].iloc[:n_train]
X_val   = df[FEATURES].iloc[n_train:n_train+n_val]
y_val   = df["isFraud"].iloc[n_train:n_train+n_val]
X_test  = df[FEATURES].iloc[n_train+n_val:]
y_test  = df["isFraud"].iloc[n_train+n_val:]

print(f"Train: {len(X_train):>7,}  · fraudes: {y_train.sum():>5,}  ({y_train.mean()*100:.2f}%)")
print(f"Val:   {len(X_val):>7,}  · fraudes: {y_val.sum():>5,}  ({y_val.mean()*100:.2f}%)")
print(f"Test:  {len(X_test):>7,}  · fraudes: {y_test.sum():>5,}  ({y_test.mean()*100:.2f}%)")

## 3. Entrenamiento del baseline

In [ ]:
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2", C=1.0,
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_SEED, n_jobs=-1
    )),
])

%time baseline.fit(X_train, y_train)
print("✔ Entrenamiento completado.")

## 4. Evaluación en test

In [ ]:
y_proba = baseline.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

metricas = {
    "ROC-AUC":   roc_auc_score(y_test, y_proba),
    "PR-AUC":    average_precision_score(y_test, y_proba),
    "Recall":    recall_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "F1":        f1_score(y_test, y_pred),
}

print("📊 Métricas del baseline en test:")
for k, v in metricas.items():
    print(f"   {k:10s} = {v:.4f}")

## 5. Visualización

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
prec, rec, _ = precision_recall_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(fpr, tpr, color='#F77F00', lw=2.5,
              label=f"AUC = {metricas['ROC-AUC']:.3f}")
axes[0].plot([0,1], [0,1], 'k--', alpha=0.4, label='Azar')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curva ROC — baseline'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, color='#1E2761', lw=2.5,
              label=f"AP = {metricas['PR-AUC']:.3f}")
axes[1].axhline(y_test.mean(), color='gray', ls='--', alpha=0.5, label='Tasa base')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva PR — baseline'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 6. Conclusiones del baseline

- **ROC-AUC ≈ 0.76:** discrimina mejor que el azar pero lejos del óptimo.
- **PR-AUC ≈ 0.18:** muy baja, indica que la mayoría de las alertas serían **falsos positivos** si usamos este modelo en producción.
- **Recall = 0.55, Precision = 0.06:** detecta el 55% de fraudes pero solo el 6% de las alertas son fraude real → impracticable enviar todo a revisión humana.

**Limitaciones identificadas:**
1. La Regresión Logística asume relación **lineal** entre features y log-odds, no captura interacciones (ej. monto alto × país riesgoso × dispositivo nuevo).
2. Aún con balanced class weight, la baja precisión hace impracticable el uso operacional.

Estas limitaciones motivan migrar a **XGBoost con Optuna** en `03_modelo_final.ipynb`.